# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json
from typing import List

import chromadb
from chromadb.utils import embedding_functions
from pydantic import BaseModel, Field
from tavily import TavilyClient

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool

In [3]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"
assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY is not set"

In [4]:
# Environment variables are loaded in the setup cell above.

# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
# TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

In [5]:
# Load the vector DB built in Part 1 (same path, collection name and embedding function)
chroma_client = chromadb.PersistentClient(path="chromadb")
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key_env_var="OPENAI_API_KEY",
    api_base=os.getenv("OPENAI_BASE_URL"),
)
collection = chroma_client.get_collection("udaplay", embedding_function=embedding_fn)
print(f"Loaded collection 'udaplay' with {collection.count()} documents")

tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

Loaded collection 'udaplay' with 15 documents


### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [6]:
@tool
def retrieve_game(query: str) -> List[dict]:
    """
    Semantic search: Finds most results in the vector DB
    args:
    - query: a question about game industry.

    You'll receive results as list. Each element contains:
    - Platform: like Game Boy, PlayStation 5, Xbox 360...
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game
    """
    try:
        results = collection.query(query_texts=[query], n_results=3)
    except Exception as e:
        return [{"error": f"Vector DB query failed: {e}"}]

    docs = []
    for metadata, distance in zip(results["metadatas"][0], results["distances"][0]):
        docs.append({
            "Platform": metadata.get("Platform"),
            "Name": metadata.get("Name"),
            "YearOfRelease": metadata.get("YearOfRelease"),
            "Description": metadata.get("Description"),
            "distance": round(distance, 4),
        })
    return docs

retrieve_game("When was Pokémon Gold and Silver released?")

[{'Platform': 'Game Boy Color',
  'Name': 'Pokémon Gold and Silver',
  'YearOfRelease': 1999,
  'Description': 'Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.',
  'distance': 0.1128},
 {'Platform': 'Game Boy Advance',
  'Name': 'Pokémon Ruby and Sapphire',
  'YearOfRelease': 2002,
  'Description': 'Third-generation Pokémon games set in the Hoenn region, featuring new Pokémon and double battles.',
  'distance': 0.1467},
 {'Platform': 'Nintendo 64',
  'Name': 'Super Mario 64',
  'YearOfRelease': 1996,
  'Description': "A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.",
  'distance': 0.2207}]

#### Evaluate Retrieval Tool

In [7]:
class EvaluationReport(BaseModel):
    useful: bool = Field(description="Whether the documents are useful to answer the question")
    description: str = Field(description="Description about the evaluation result")


@tool
def evaluate_retrieval(question: str, retrieved_docs: List[dict]) -> dict:
    """
    Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.
    args:
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    """
    judge = LLM(model="gpt-4o-mini", temperature=0.0)
    prompt = (
        "Your task is to evaluate if the documents are enough to respond the query. "
        "Give a detailed explanation, so it's possible to take an action to accept it or not.\n\n"
        f"Question: {question}\n\n"
        f"Retrieved documents:\n{json.dumps(retrieved_docs, indent=2, ensure_ascii=False)}"
    )
    try:
        response = judge.invoke(prompt, response_format=EvaluationReport)
        report = EvaluationReport.model_validate_json(response.content)
    except Exception as e:
        return {"useful": False, "description": f"Evaluation failed: {e}"}
    return report.model_dump()

evaluate_retrieval(
    "When was Pokémon Gold and Silver released?",
    retrieve_game("When was Pokémon Gold and Silver released?"),
)

{'useful': True,
 'description': 'The retrieved documents contain relevant information regarding the release date of Pokémon Gold and Silver. Specifically, the first document explicitly states that Pokémon Gold and Silver were released in 1999 for the Game Boy Color. This directly answers the query about the release date of the game. The other documents, while related to different Pokémon games, do not provide any conflicting information and are not necessary for answering the question. Therefore, the documents are sufficient to respond to the query.'}

#### Game Web Search Tool

In [8]:
@tool
def game_web_search(question: str) -> dict:
    """
    Web search: searches the web with Tavily when the vector DB results aren't good enough
    args:
    - question: a question about game industry.

    Returns the search results, including each page's URL, so the answer can cite its source.
    """
    try:
        response = tavily_client.search(
            query=question,
            search_depth="advanced",
            max_results=5,
            include_answer=True,
        )
    except Exception as e:
        return {"error": f"Web search failed: {e}"}

    return {
        "answer": response.get("answer"),
        "results": [
            {"title": r.get("title"), "url": r.get("url"), "content": r.get("content")}
            for r in response.get("results", [])
        ],
    }

### Agent

In [9]:
instructions = """
You are UdaPlay, an AI research agent specialized in the video game industry.

Follow this workflow for every question:
1. ALWAYS call `retrieve_game` first to search the internal game database.
2. Call `evaluate_retrieval` with the user's question and the retrieved documents.
3. If the evaluation says the documents are useful, answer from them.
   Otherwise, call `game_web_search` to find the information on the web.
4. Never make up facts. If neither source has the answer, say so.

Answer format:
- Give a clear, concise answer to the question.
- End with a "Source:" line: "Internal database" or the URL(s) of the web pages you used.
"""

agent = Agent(
    model_name="gpt-4o-mini",
    instructions=instructions,
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    temperature=0.0,
)

In [10]:
def ask(question: str, session_id: str = "udaplay"):
    run = agent.invoke(question, session_id=session_id)
    final_state = run.get_final_state()
    used_tools = [
        m.name for m in final_state["messages"] if isinstance(m, ToolMessage)
    ]
    print(f"Q: {question}")
    print(f"Tools used (whole session): {used_tools}")
    print(f"A: {final_state['messages'][-1].content}\n")
    return run

questions = [
    "When was Pokémon Gold and Silver released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for PlayStation 5?",
]
for q in questions:
    ask(q, session_id=q)  # separate session per question

# Conversation state: a follow-up in the same session
ask("And what platform was it released on?", session_id=questions[1])

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


...

[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Q: Was Mortal Kombat X released for PlayStation 5?
Tools used (whole session): ['retrieve_game', 'evaluate_retrieval', 'game_web_search']
A: Mortal Kombat X was originally released for PlayStation 4 on April 14, 2015. While it is playable on PlayStation 5, it was not specifically released for that platform. However, it is included in the PlayStation Plus Collection, making it available to PS5 users who subscribe to PlayStation Plus. 

Source: [Mortal Kombat Online](https://www.mortalkombatonline.com/t/mkx/ps5-playstation-plus-collection-to-include-mortal-kombat-x/85LJ3iL2YW7V)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Q: And what platform was it released on?
Tools used (whole session): ['retrieve_game', 'evaluate_retrieval']
A: "Super Mario 64" was released on the Nintendo 64 platform. 

Source: Internal database



Run('c16783c9-5594-4916-8d32-dd69e0f09a97')

### (Optional) Advanced — Stand Out

In [11]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes